In [1]:
import torch
import torch.nn as nn
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
import numpy as np
import time
import torch
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

In [2]:
# 1. Reproduksibilitas (Protokol 4.2)
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

# 2. Dimensi Input & Normalisasi ImageNet (Protokol 4.2)
# Ini untuk B1, B2, B3, dan B4 (Tanpa Augmentasi)
transform_standard = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
selected_classes = [0, 1, 2, 3, 4, 11, 14, 15, 28, 29]
class_mapping = {orig_label: new_label for new_label, orig_label in enumerate(selected_classes)}

def map_label(label):
    return class_mapping.get(label, label)

In [3]:
#dataset
ds_trainval = datasets.OxfordIIITPet(root='./data', split='trainval', download=True, transform=transform_standard, target_transform=map_label)

# PERBAIKAN: Tambahkan target_transform=map_label pada ds_test
ds_test = datasets.OxfordIIITPet(root='./data', split='test', download=True, transform=transform_standard, target_transform=map_label)

In [4]:
# Menggabungkan seluruh dataset
full_dataset = torch.utils.data.ConcatDataset([ds_trainval, ds_test])

# Ambil target/label dari masing-masing split untuk mempermudah filtering
targets = ds_trainval._labels + ds_test._labels

# Cari indeks gambar yang masuk ke dalam 10 kelas tersebut
filtered_indices = [i for i, label in enumerate(targets) if label in selected_classes]
filtered_targets = [targets[i] for i in filtered_indices]

# 5. Stratified Split: 70% Train, 15% Val, 15% Test
# Split pertama: Pisahkan Train (70%) dan Sisa (30%)
train_idx, temp_idx, train_targets, temp_targets = train_test_split(
    filtered_indices, filtered_targets, test_size=0.30, stratify=filtered_targets, random_state=seed
)

# Split kedua: Bagi sisa (30%) menjadi Val (15%) dan Test (15%)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, stratify=temp_targets, random_state=seed
)

# Buat subset berdasarkan indeks yang sudah di-split
train_set = Subset(full_dataset, train_idx)
val_set = Subset(full_dataset, val_idx)
test_set = Subset(full_dataset, test_idx)

def cache(subset):
    xs, ys = zip(*[(x, y) for x, y in subset])   # decode + resize once
    return torch.utils.data.TensorDataset(torch.stack(xs), torch.tensor(ys))

train_set, val_set, test_set = cache(train_set), cache(val_set), cache(test_set)

train_loader = DataLoader(train_set, batch_size=32, shuffle=True,  pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=32, shuffle=False, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=32, shuffle=False, pin_memory=True)

print(f"Total Train images: {len(train_set)}")
print(f"Total Val images: {len(val_set)}")
print(f"Total Test images: {len(test_set)}")

Total Train images: 1391
Total Val images: 298
Total Test images: 299


In [5]:
weights = models.ResNet18_Weights.IMAGENET1K_V1
model = models.resnet18(weights=weights)

# 2. Konfigurasi B2: Feature Extraction (Bekukan Backbone)
for param in model.parameters():
    param.requires_grad = False
# Ganti classifier head untuk 10 kelas
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 10)

# 3. Konfigurasi B3: Unfreeze Blok Residual Terakhir (layer4)
for param in model.layer4.parameters():
    param.requires_grad = True
# Optimasi dengan Differential Learning Rate
optimizer = torch.optim.AdamW([
    {"params": model.layer4.parameters(), "lr": 1e-4},
    {"params": model.fc.parameters(),
    "lr": 1e-3},
], weight_decay=1e-2)

# 4. Pipeline Transformasi Data
# Data latih: menerapkan augmentasi aktif
train_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=weights.transforms().mean,
    std=weights.transforms().std),
    ])
# Data validasi/uji: hanya deterministik tanpa distorsi

eval_transforms = weights.transforms()
# Verifikasi parameter yang dapat dilatih

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable Parameters: {trainable_params:,} / {total_params:,}")

Trainable Parameters: 8,398,858 / 11,181,642


In [6]:
# ==========================================
# 1. Fungsi Ekstraksi Parameter & Learning Rate
# ==========================================
def get_model_summary(model, optimizer):
    # Hitung jumlah parameter yang dapat dilatih
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    # Ambil learning rate untuk setiap kelompok parameter
    lr_groups = [group['lr'] for group in optimizer.param_groups]
    
    return trainable_params, lr_groups

# ==========================================
# 2. Fungsi Evaluasi Akurasi dan Macro F1
# ==========================================
def evaluate_metrics(model, dataloader, device):
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
            
    # Hitung akurasi dan Macro F1-Score menggunakan sklearn
    accuracy = accuracy_score(all_targets, all_preds)
    macro_f1 = f1_score(all_targets, all_preds, average='macro')
    
    return accuracy, macro_f1

# ==========================================
# 3. Training Loop dengan Pencatatan Penuh & Early Stopping
# ==========================================
def train_and_log_experiment(model, train_loader, val_loader, test_loader, optimizer, criterion, device, max_epochs=50, patience=5):
    trainable_params, lr_groups = get_model_summary(model, optimizer)
    
    # Variabel untuk pencatatan metrik
    start_time = time.time()
    best_val_loss = float('inf')
    epochs_no_improve = 0
    converged_epoch = max_epochs
    best_model_state = None

    model.to(device)
    
    for epoch in range(max_epochs):
        model.train()
        train_loss = 0.0
        
        # Training pass
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * inputs.size(0)
            
        train_loss = train_loss / len(train_loader.dataset)
        
        # Validation pass (Hitung Loss untuk Early Stopping)
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item() * inputs.size(0)
                
        val_loss = val_loss / len(val_loader.dataset)
        
        print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
        
        # Early Stopping Logic (berdasarkan validation loss)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            best_model_state = model.state_dict() # Simpan model terbaik
        else:
            epochs_no_improve += 1
            
        if epochs_no_improve >= patience:
            converged_epoch = epoch + 1
            print(f"Early stopping dipicu pada epoch {converged_epoch}!")
            break
            
    # Kembalikan bobot model ke performa terbaik sebelum overfitting
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        
    total_duration = time.time() - start_time
    
    # Evaluasi akhir pada Validation Set
    val_acc, val_f1 = evaluate_metrics(model, val_loader, device)
    
    # Evaluasi akhir pada Test Set
    test_acc, test_f1 = evaluate_metrics(model, test_loader, device)
    
    # Kompilasi hasil dalam satu dictionary
    experiment_results = {
        "Trainable Parameters": trainable_params,
        "Learning Rate(s)": lr_groups,
        "Epochs to Converge": converged_epoch,
        "Total Training Time (s)": round(total_duration, 2),
        "Final Validation Accuracy": round(val_acc, 4),
        "Final Test Macro F1-Score": round(test_f1, 4),
        "Final Test Accuracy": round(test_acc, 4)
    }
    
    return experiment_results

In [7]:
model_b1 = models.resnet18(weights=None)
model_b1.fc = nn.Linear(model_b1.fc.in_features, 10)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = torch.nn.CrossEntropyLoss()

# Semua parameter dilatih
optimizer_b1 = torch.optim.AdamW(model_b1.parameters(), lr=1e-3, weight_decay=1e-2)

hasil_b1 = train_and_log_experiment(
    model=model_b1,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    optimizer=optimizer_b1,
    criterion=criterion,
    device=device,
    max_epochs=10,   
    patience=4       # Patience early stopping 3-5 sesuai panduan
)

# Cetak rekap metrik
print("\n--- Rekapitulasi Metrik Eksperimen B1 ---")
for key, value in hasil_b1.items():
    print(f"{key}: {value}")

Epoch 01 | Train Loss: 2.3054 | Val Loss: 2.2582
Epoch 02 | Train Loss: 2.0048 | Val Loss: 2.2271
Epoch 03 | Train Loss: 1.8288 | Val Loss: 2.3388
Epoch 04 | Train Loss: 1.6994 | Val Loss: 1.9368
Epoch 05 | Train Loss: 1.5457 | Val Loss: 1.7457
Epoch 06 | Train Loss: 1.4182 | Val Loss: 2.9371
Epoch 07 | Train Loss: 1.3260 | Val Loss: 1.9869
Epoch 08 | Train Loss: 1.2108 | Val Loss: 2.1818
Epoch 09 | Train Loss: 1.0855 | Val Loss: 1.7862
Early stopping dipicu pada epoch 9!

--- Rekapitulasi Metrik Eksperimen B1 ---
Trainable Parameters: 11181642
Learning Rate(s): [0.001]
Epochs to Converge: 9
Total Training Time (s): 33.43
Final Validation Accuracy: 0.4027
Final Test Macro F1-Score: 0.3931
Final Test Accuracy: 0.4181


In [8]:
# ==========================================
# Eksperimen B2: Feature Extraction
# ==========================================
weights = models.ResNet18_Weights.IMAGENET1K_V1
model_b2 = models.resnet18(weights=weights)

# Bekukan seluruh lapisan backbone
for param in model_b2.parameters():
    param.requires_grad = False
    
# Ganti classifier head
model_b2.fc = nn.Linear(model_b2.fc.in_features, 10)

# Optimizer HANYA untuk parameter fc
optimizer_b2 = torch.optim.AdamW(model_b2.fc.parameters(), lr=1e-3, weight_decay=1e-2)

hasil_b2 = train_and_log_experiment(
    model=model_b2,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    optimizer=optimizer_b2,
    criterion=criterion,
    device=device,
    max_epochs=10,   
    patience=4
)

print("\n--- Rekapitulasi Metrik Eksperimen B2 ---")
for key, value in hasil_b2.items():
    print(f"{key}: {value}")

Epoch 01 | Train Loss: 1.4575 | Val Loss: 0.7090
Epoch 02 | Train Loss: 0.5747 | Val Loss: 0.4489
Epoch 03 | Train Loss: 0.3923 | Val Loss: 0.3407
Epoch 04 | Train Loss: 0.3226 | Val Loss: 0.3198
Epoch 05 | Train Loss: 0.2731 | Val Loss: 0.2727
Epoch 06 | Train Loss: 0.2450 | Val Loss: 0.2593
Epoch 07 | Train Loss: 0.2157 | Val Loss: 0.2450
Epoch 08 | Train Loss: 0.1922 | Val Loss: 0.2381
Epoch 09 | Train Loss: 0.1837 | Val Loss: 0.2343
Epoch 10 | Train Loss: 0.1691 | Val Loss: 0.2204

--- Rekapitulasi Metrik Eksperimen B2 ---
Trainable Parameters: 5130
Learning Rate(s): [0.001]
Epochs to Converge: 10
Total Training Time (s): 14.65
Final Validation Accuracy: 0.9362
Final Test Macro F1-Score: 0.9168
Final Test Accuracy: 0.9164


In [9]:
# ==========================================
# Eksperimen B3: Progressive Fine-Tuning
# ==========================================
weights = models.ResNet18_Weights.IMAGENET1K_V1
model_b3 = models.resnet18(weights=weights)

# Bekukan seluruh backbone di awal
for param in model_b3.parameters():
    param.requires_grad = False
    
# Buka pembekuan (unfreeze) hanya pada blok residual terakhir (layer4)
for param in model_b3.layer4.parameters():
    param.requires_grad = True
    
# Ganti classifier head
model_b3.fc = nn.Linear(model_b3.fc.in_features, 10)

# Optimizer dengan Differential Learning Rate
optimizer_b3 = torch.optim.AdamW([
    {"params": model_b3.layer4.parameters(), "lr": 1e-4},
    {"params": model_b3.fc.parameters(), "lr": 1e-3},
], weight_decay=1e-2)

hasil_b3 = train_and_log_experiment(
    model=model_b3,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    optimizer=optimizer_b3,
    criterion=criterion,
    device=device,
    max_epochs=10,   
    patience=4
)

print("\n--- Rekapitulasi Metrik Eksperimen B3 ---")
for key, value in hasil_b3.items():
    print(f"{key}: {value}")

Epoch 01 | Train Loss: 0.7363 | Val Loss: 0.2601
Epoch 02 | Train Loss: 0.1307 | Val Loss: 0.2213
Epoch 03 | Train Loss: 0.0448 | Val Loss: 0.2076
Epoch 04 | Train Loss: 0.0161 | Val Loss: 0.1938
Epoch 05 | Train Loss: 0.0096 | Val Loss: 0.1924
Epoch 06 | Train Loss: 0.0077 | Val Loss: 0.1948
Epoch 07 | Train Loss: 0.0045 | Val Loss: 0.1899
Epoch 08 | Train Loss: 0.0036 | Val Loss: 0.1936
Epoch 09 | Train Loss: 0.0039 | Val Loss: 0.1920
Epoch 10 | Train Loss: 0.0023 | Val Loss: 0.2019

--- Rekapitulasi Metrik Eksperimen B3 ---
Trainable Parameters: 8398858
Learning Rate(s): [0.0001, 0.001]
Epochs to Converge: 10
Total Training Time (s): 18.01
Final Validation Accuracy: 0.9396
Final Test Macro F1-Score: 0.9261
Final Test Accuracy: 0.9264


In [10]:
# ==========================================
# Eksperimen B4 (Skenario 1): TANPA Augmentasi
# ==========================================
model_b4_no_aug = models.resnet18(weights=weights)
for param in model_b4_no_aug.parameters():
    param.requires_grad = False
model_b4_no_aug.fc = nn.Linear(model_b4_no_aug.fc.in_features, 10)
optimizer_b4_no_aug = torch.optim.AdamW(model_b4_no_aug.fc.parameters(), lr=1e-3, weight_decay=1e-2)

print("\n=== Memulai B4 Skenario 1 (Tanpa Augmentasi) ===")
hasil_b4_no_aug = train_and_log_experiment(
    model=model_b4_no_aug,
    train_loader=train_loader, # Gunakan dataloader tanpa augmentasi
    val_loader=val_loader,
    test_loader=test_loader,
    optimizer=optimizer_b4_no_aug,
    criterion=criterion,
    device=device,
    max_epochs=10,   
    patience=4
)



=== Memulai B4 Skenario 1 (Tanpa Augmentasi) ===
Epoch 01 | Train Loss: 1.4305 | Val Loss: 0.7114
Epoch 02 | Train Loss: 0.5539 | Val Loss: 0.4452
Epoch 03 | Train Loss: 0.3872 | Val Loss: 0.3499
Epoch 04 | Train Loss: 0.3088 | Val Loss: 0.3100
Epoch 05 | Train Loss: 0.2731 | Val Loss: 0.2708
Epoch 06 | Train Loss: 0.2282 | Val Loss: 0.2576
Epoch 07 | Train Loss: 0.2126 | Val Loss: 0.2493
Epoch 08 | Train Loss: 0.1899 | Val Loss: 0.2314
Epoch 09 | Train Loss: 0.1787 | Val Loss: 0.2350
Epoch 10 | Train Loss: 0.1684 | Val Loss: 0.2247


In [11]:
train_transforms_aug = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 2. Muat ulang dataset khusus augmentasi
ds_trainval_aug = datasets.OxfordIIITPet(root='./data', split='trainval', download=True, transform=train_transforms_aug, target_transform=map_label)
ds_test_aug = datasets.OxfordIIITPet(root='./data', split='test', download=True, transform=train_transforms_aug, target_transform=map_label)
full_dataset_aug = torch.utils.data.ConcatDataset([ds_trainval_aug, ds_test_aug])

train_set_aug = Subset(full_dataset_aug, train_idx)

# KUNCI UTAMA AGAR TIDAK STUCK DI WINDOWS: Set num_workers=0 dan persistent_workers=False
train_loader_aug = DataLoader(
    train_set_aug, 
    batch_size=32, 
    shuffle=True, 
    num_workers=0,        # Wajib 0 di Windows untuk menghindari deadlock
    pin_memory=False      # Matikan pin_memory jika num_workers 0
)

print(f"Total Train images (Augmented): {len(train_set_aug)}")

# ==========================================
# Jalankan Eksperimen B4 Skenario 2 (Dengan Augmentasi)
# ==========================================
model_b4_aug = models.resnet18(weights=weights)
for param in model_b4_aug.parameters():
    param.requires_grad = False
model_b4_aug.fc = nn.Linear(model_b4_aug.fc.in_features, 10)
optimizer_b4_aug = torch.optim.AdamW(model_b4_aug.fc.parameters(), lr=1e-3, weight_decay=1e-2)

print("\n=== Memulai B4 Skenario 2 (Dengan Augmentasi Lengkap sesuai Soal) ===")
hasil_b4_aug = train_and_log_experiment(
    model=model_b4_aug,
    train_loader=train_loader_aug, 
    val_loader=val_loader,
    test_loader=test_loader,
    optimizer=optimizer_b4_aug,
    criterion=criterion,
    device=device,
    max_epochs=10,   
    patience=4
)

print("\n--- Rekapitulasi Metrik Eksperimen B4 (Dengan Augmentasi) ---")
for key, value in hasil_b4_aug.items():
    print(f"{key}: {value}")

Total Train images (Augmented): 1391

=== Memulai B4 Skenario 2 (Dengan Augmentasi Lengkap sesuai Soal) ===
Epoch 01 | Train Loss: 1.4311 | Val Loss: 0.7727
Epoch 02 | Train Loss: 0.6411 | Val Loss: 0.4513
Epoch 03 | Train Loss: 0.4567 | Val Loss: 0.3781
Epoch 04 | Train Loss: 0.3748 | Val Loss: 0.3231
Epoch 05 | Train Loss: 0.3328 | Val Loss: 0.3045
Epoch 06 | Train Loss: 0.3116 | Val Loss: 0.2894
Epoch 07 | Train Loss: 0.2611 | Val Loss: 0.2729
Epoch 08 | Train Loss: 0.2523 | Val Loss: 0.2590
Epoch 09 | Train Loss: 0.2370 | Val Loss: 0.2395
Epoch 10 | Train Loss: 0.2273 | Val Loss: 0.2489

--- Rekapitulasi Metrik Eksperimen B4 (Dengan Augmentasi) ---
Trainable Parameters: 5130
Learning Rate(s): [0.001]
Epochs to Converge: 10
Total Training Time (s): 100.89
Final Validation Accuracy: 0.9128
Final Test Macro F1-Score: 0.9034
Final Test Accuracy: 0.903


In [12]:
# ==========================================
# Eksperimen B5: Arsitektur Pembanding
# ==========================================
# Inisialisasi EfficientNet-B0
weights_b5 = models.EfficientNet_B0_Weights.IMAGENET1K_V1
model_b5 = models.efficientnet_b0(weights=weights_b5)

# Bekukan seluruh backbone untuk feature extraction baseline
for param in model_b5.parameters():
    param.requires_grad = False
    
# Classifier head pada EfficientNet ada di model.classifier[1]
model_b5.classifier[1] = nn.Linear(model_b5.classifier[1].in_features, 10)

# Optimizer hanya untuk layer classifier
optimizer_b5 = torch.optim.AdamW(model_b5.classifier[1].parameters(), lr=1e-3, weight_decay=1e-2)

hasil_b5 = train_and_log_experiment(
    model=model_b5,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    optimizer=optimizer_b5,
    criterion=criterion,
    device=device,
    max_epochs=10,   
    patience=4
)

print("\n--- Rekapitulasi Metrik Eksperimen B5 ---")
for key, value in hasil_b5.items():
    print(f"{key}: {value}")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\annab/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:03<00:00, 5.49MB/s]


Epoch 01 | Train Loss: 1.5351 | Val Loss: 0.8267
Epoch 02 | Train Loss: 0.7399 | Val Loss: 0.5237
Epoch 03 | Train Loss: 0.5262 | Val Loss: 0.4273
Epoch 04 | Train Loss: 0.4313 | Val Loss: 0.3855
Epoch 05 | Train Loss: 0.3766 | Val Loss: 0.3511
Epoch 06 | Train Loss: 0.3258 | Val Loss: 0.3222
Epoch 07 | Train Loss: 0.2944 | Val Loss: 0.3127
Epoch 08 | Train Loss: 0.2693 | Val Loss: 0.2898
Epoch 09 | Train Loss: 0.2396 | Val Loss: 0.2914
Epoch 10 | Train Loss: 0.2292 | Val Loss: 0.2849

--- Rekapitulasi Metrik Eksperimen B5 ---
Trainable Parameters: 12810
Learning Rate(s): [0.001]
Epochs to Converge: 10
Total Training Time (s): 19.0
Final Validation Accuracy: 0.9027
Final Test Macro F1-Score: 0.9068
Final Test Accuracy: 0.9064
